Scraping free patterns from Herrschners

In [1]:
import requests
import re
import time
import pandas as pd
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

def scrape_product(product_link):

    r = requests.get(product_link, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    row = {}

    # Getting the info (which contains project type, craft type, and difficulty level, but not all postings)
    names = soup.select(".productView-info-name")
    values = soup.select(".productView-info-value")
    for name, value in zip(names, values):
        key = name.get_text(strip=True).lower().replace(" ", "_")
        row[key] = value.get_text(strip=True)
    
    merged_info = " ".join(v.get_text(strip=True) for v in values)
    row["info"] = merged_info

    # Getting description, skill level, size, materials, etc. from the description section)
    desc = soup.select_one(".productView-desc-content")
    if desc:
        row["description"] = desc.get_text(" ", strip=True)

    # Getting review count by extracting the number from the review link text
    review = soup.select_one(".productView-reviewLink")
    # print (review.get_text() if review else "No review link found")

    if review:
        match = re.search(r"\d+", review.get_text())
        row["review_count"] = int(match.group()) if match else 0
    else:
        # if text is no reviews yet
        row["review_count"] = 0

    # Counting number of stars by counting the number of full star icons
    rating_div = soup.select_one(".productView-rating")

    if rating_div:
        stars = rating_div.select(".icon.icon--ratingFull")
        row["star_rating"] = len(stars)
    else:
        row["star_rating"] = 0

    return row

In [2]:
base_url = "https://herrschners.com/herrschners/knit-crochet/books-patterns/free-pattern-downloads/"
rows = []
page = 1

while True:
    print(f"Scraping page {page}...")
    if page == 1:
        url = base_url
    else:
        url = f"{base_url}?page={page}"

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.select("article.card")
    if not cards:
        print("No more products found. Ending scrape.")
        break

    for card in cards:
        product_link = card.get("data-href")
        img = card.select_one("img")
        row = {
            "title": img.get("title") if img else None,
            "image": img["src"] if img else None,
            "product_link": product_link
        }
        if product_link:
            features = scrape_product(product_link)
            row.update(features)
        rows.append(row)
        time.sleep(0.5)

    page += 1

df = pd.DataFrame(rows)

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
No more products found. Ending scrape.


In [3]:
print (df.shape)
df.head(10)

(548, 18)


,title,image,product_link,sku:,upc:,project_type:,skill:,difficulty:,info,description,review_count,star_rating,occasion:,season:,theme:,fiber_content:,yarn_weight:,thread_size:
0,Village Yarn Blue Burst Pot Holders Free Download,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/village-yarn-blue-burs...,F01729,,Pot Holders,Crochet,Easy,F01729 Pot Holders Crochet Easy,Brighten your kitchen with a splash of color! ...,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,Willow Yarns Garden Song Shawlette Free Download,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/willow-yarns-garden-so...,F01728,,Shawl,Knit,Easy,F01728 Shawl Knit Easy,Be ready for backyard garden parties with this...,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,Herrschners English Garden Doily Free Download,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/herrschners-english-ga...,F01727,,Doily,Crochet,Easy,F01727 Doily Crochet Easy,No green thumb is needed to grow a garden of t...,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,Herrschners Shamrock Glow Blanket Free Download,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/herrschners-shamrock-g...,F01726,,Blanket,Crochet,Easy,F01726 St. Patrick's Day Blanket Crochet Easy,It’s your lucky day! This free easy crochet bl...,0,0,St. Patrick's Day,NaN,NaN,NaN,NaN,NaN
4,Herrschners Tilework Bottle Carriers - Set of ...,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/herrschners-tilework-b...,F01725,,Fashion Accessories & Novelties,Crochet,Intermediate,F01725 Fashion Accessories & Novelties Croche...,Stay hydrated on the go with these cute and co...,1,5,NaN,NaN,NaN,NaN,NaN,NaN
5,Willow Yarns Muted Melody Cowl Free Download,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/willow-yarns-muted-mel...,F01724,,Cowl,Crochet,Intermediate,F01724 Cowl Crochet Intermediate,Cozy up to colorwork with this exquisitely det...,1,4,NaN,NaN,NaN,NaN,NaN,NaN
6,Willow Yarns Lovey Dovey Dishcloths Free Download,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/willow-yarns-lovey-dov...,F01723,,Dishcloth,Crochet,Intermediate,F01723 Dishcloth Crochet Intermediate,Add an extra dash of love to the heart of your...,2,5,NaN,NaN,NaN,NaN,NaN,NaN
7,Soho Sugar Sky Earflap Hat Free Download,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/soho-sugar-sky-earflap...,F01722,,Hat,Crochet,Easy,F01722 Hat Crochet Easy,Keep cozy and colorful on cold-weather days. T...,0,0,NaN,NaN,NaN,NaN,NaN,NaN
8,Herrschners Stitch & Switch Tote Free Download,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/herrschners-stitch-swi...,F01721,,Bags & Totes,Crochet,Easy,F01721 Bags & Totes Crochet Easy,Be tote-ally on trend with this granny square ...,2,5,NaN,NaN,NaN,NaN,NaN,NaN
9,Herrschners Fin-tastic Purse Free Download,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ste...,https://herrschners.com/herrschners-fin-tastic...,F01708,,Tote,Crochet,Intermediate,F01708 Tote Crochet Intermediate,Swim around town with a fishy friend to carry ...,1,5,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.to_csv("herrschners_crochet_patterns.csv", index=False)